# Week 10 – Phân cụm và đối chiếu với nhận xét Tuần 9 (Nhóm 07)
**Nhóm 07 – lớp 62FIT4MLA – CLC01.** Notebook này gom các việc Tuần 9 (theo nhận xét của thầy Khánh) và nội dung Tuần 10 (phân cụm, nối với Gauss và hiệp phương sai).

**Đối chiếu nhận xét Tuần 9**

| Yêu cầu của thầy | Mục | Trạng thái |
|---|---|---|
| 1. In ID–text–label trước fit; kiểm lệch hàng; kiểm fold áp đúng X, y | 2 | Đã thêm; chạy trên dữ liệu dự án |
| 1b. Giải thích kết quả TF-IDF 0,841 và 0,336 | 3 | Đã thêm, có bằng chứng; khác với nhận định của thầy (xem mục 3) |
| 2. Cố định seed, lưu danh sách fold, chạy lại phải khớp | 2 | Đã thêm (`folds_w10.json`) |
| 3. Giữ bảng log loss LR; NB xuống phụ lục | Notebook W9 | Không thuộc Tuần 10; NB nằm ở notebook W9 |
| 4. Trích dẫn Lee et al. (2021), bỏ Pan nếu không tìm được | 9 | Đã xác minh cả hai; cách chia của Lee et al. còn cần đọc lại |
| Đọc ≥ 10 ca nhầm, bắt đầu từ WNL Glastonbury; tách lỗi tiền xử lý với độ khó | 6 | Đã thêm mẫu ghi chú; cần các em tự đọc và điền |
| So xếp hạng trong bài với gán mức tuyệt đối | 7 | Đã thêm |
| 5 văn bản ngoài, nhãn gán độc lập trước khi chạy mô hình | 8 | Đã thêm; cần các em điền file nhãn |
| BERT không bắt buộc | 10 | Không thực hiện trong notebook này |
| Hai câu hỏi tự trả lời | 10 | Đã thêm |

## 0. Chuẩn bị

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)   # cố định seed để kết quả tái lập được

## 1. Ôn lại Tuần 9 (ví dụ slide, kiểm tra bằng code)

In [ ]:
class0 = np.array([[4,3],[6,7],[8,5],[10,9]], dtype=float)     # D1..D4
class1 = np.array([[12,11],[14,15],[16,13],[18,17]], dtype=float)  # P1..P4

mu0, mu1 = class0.mean(axis=0), class1.mean(axis=0)
assert np.allclose(mu0, [7, 6]) and np.allclose(mu1, [15, 14])

def sq_dist(x, c):
    # D² = (x1 - c1)² + (x2 - c2)², như slide Tuần 9 Part 1, slide 34
    return ((x - c) ** 2).sum(axis=-1)

N1, N2 = np.array([11, 12.]), np.array([9, 11.])
assert sq_dist(N1, mu0) == 52 and sq_dist(N1, mu1) == 20
assert sq_dist(N2, mu0) == 29 and sq_dist(N2, mu1) == 45

# Phương sai mẫu chia cho n (như slide): fractions lớp 0 = 5
assert np.isclose(np.var(class0[:, 0]), 5.0)
print("Các phép kiểm khớp slide Tuần 9.")

### 1b. Hai ý khác từ slide Tuần 9
Mật độ là chiều cao, xác suất là diện tích (slide Part 1). Thiếu dữ liệu không phải 0 (slide Part 1 và Part 2).

In [ ]:
from scipy.stats import norm
sd = np.sqrt(5.0)
assert norm.pdf(6, 7, sd) > norm.pdf(10, 7, sd)          # 6 gần µ hơn 10
prob_6_8 = norm.cdf(8, 7, sd) - norm.cdf(6, 7, sd)        # diện tích = xác suất
print('P(6 ≤ X ≤ 8) =', round(prob_6_8, 3), '(diện tích, không phải chiều cao)')

decimals_resp = np.array([14.0, np.nan, 9.0, 12.0])
mean_observed = np.nanmean(decimals_resp)
mean_if_zero = np.nan_to_num(decimals_resp).mean()
print('Trung bình trên quan sát được:', round(mean_observed, 2), '| nếu điền 0 (sai):', round(mean_if_zero, 2))
assert not np.isclose(mean_observed, mean_if_zero)

## 2. Việc 1: kiểm tra trước khi fit
Trước khi fit, in mẫu ID–text–label, kiểm hai bảng có cùng thứ tự hàng, và cố định fold. Nếu hàng lệch, mọi số liệu sau đó đều sai. Phần này chỉ chạy khi notebook dự án đã có `data`, `df`, `EXP`.

In [ ]:
import os, json
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

LEVELS_W10 = ['Elementary', 'Intermediate', 'Advanced']
ORD_W10 = {l: i for i, l in enumerate(LEVELS_W10)}
FOLD_FILE = 'folds_w10.json'

if 'data' not in globals() or 'df' not in globals() or 'EXP' not in globals():
    print('Bỏ qua mục 2, 3, 6, 7, 8: cần data, df, EXP từ Group07_Notebook.ipynb. Mở notebook đó, chạy đến hết mục 8, rồi chạy lại notebook này.')
else:
    # 2a. Hai bảng phải cùng thứ tự hàng
    assert len(data) == len(df), 'Số hàng khác nhau giữa văn bản và đặc trưng'
    assert (data.article.values == df.article.values).all() and (data.level.values == df.level.values).all(), \
        'LỆCH HÀNG giữa văn bản và đặc trưng: sửa trước khi làm mọi việc khác'
    texts = data.text.values
    y = df.level.map(ORD_W10).values
    groups = df.article.values
    assert len(texts) == len(y) == len(groups) == len(df)

    # 2b. In mẫu ID–text–label trước fit
    print('Mẫu ID | text | nhãn (trước fit):')
    for i in [0, 1, 2]:
        print(f'  {df.article.iloc[i]} | {df.level.iloc[i]} -> nhãn {y[i]} | {texts[i][:60]!r}')

    # 2c. Fold cố định; lưu ra file; chạy lại phải khớp
    def make_folds(seed=0):
        return [{'train': tr.tolist(), 'test': te.tolist()}
                for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), y, groups)]
    folds = make_folds(0)
    cover = np.zeros(len(y), int)
    for f in folds:
        assert not set(groups[f['train']]) & set(groups[f['test']]), 'Một bài nằm ở cả train và test'
        cover[f['test']] += 1
    assert (cover == 1).all(), 'Mỗi văn bản phải nằm đúng một lần trong test'
    if os.path.exists(FOLD_FILE):
        assert json.load(open(FOLD_FILE, encoding='utf-8')) == folds, 'Fold khác file đã lưu: kiểm tra seed và dữ liệu'
        print('Fold khớp với', FOLD_FILE)
    else:
        json.dump(folds, open(FOLD_FILE, 'w', encoding='utf-8'))
        print('Đã lưu fold ra', FOLD_FILE)

## 3. Việc 1 (tiếp): vì sao chia ngẫu nhiên cho 0,336
Nhận xét của thầy: chia ngẫu nhiên để ba phiên bản của cùng một bài nằm ở cả hai phía, nên accuracy chỉ có thể **cao hơn hoặc bằng** chia theo bài. Lý do: láng giềng gần nhất của một bản thường là bản khác của cùng bài, nên khi chia ngẫu nhiên, mô hình học được từ vựng riêng của bài rồi nhận ra nó ở bản còn lại (rò rỉ theo bài).

Kết quả của nhóm đi ngược chiều: chia ngẫu nhiên 0,336 thấp hơn chia theo bài 0,841. Trước hết kiểm tra căn chỉnh (ô 8): nếu ô 8 báo hai bảng cùng thứ tự hàng và fold áp đúng X, y thì không phải lỗi lệch hàng. Phần còn lại được giải thích bằng hiệu ứng cùng bài: láng giềng gần nhất của mỗi bản thường là bản anh em có nhãn khác (ô 10 đo khoảng 98% cùng bài, khoảng 1% cùng mức). Mô hình học từ vựng của bài nên đoán ngược mức của bản anh em, nên chia ngẫu nhiên thấp hơn cả mức đoán bừa. Nếu ô 8 đúng, đây là đặc điểm của dữ liệu và cách chia, không phải lỗi mã; nhưng nó bác bỏ nhận định “chia ngẫu nhiên chỉ có thể cao hơn hoặc bằng”, cần trao đổi với thầy.

Chỉ ghi số liệu vào báo cáo sau khi ô 10 chạy lại trên dữ liệu đã sửa. Điều này cần trao đổi với thầy trước khi viết vào báo cáo.

In [ ]:
if 'folds' in globals():
    M = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
    S = cosine_similarity(M); np.fill_diagonal(S, -1)
    nn = S.argmax(axis=1)
    print('Láng giềng gần nhất cùng bài:', round((groups[nn] == groups).mean(), 3))
    print('Láng giềng gần nhất cùng mức:', round((y[nn] == y).mean(), 3))

    rand_folds = [{'train': tr.tolist(), 'test': te.tolist()}
                  for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(texts, y)]
    def tfidf_acc(fs):
        accs = []
        for f in fs:
            tr, te = np.array(f['train']), np.array(f['test'])
            m = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                              LogisticRegression(C=30, max_iter=3000)).fit(texts[tr], y[tr])
            accs.append(accuracy_score(y[te], m.predict(texts[te])))
        return float(np.mean(accs))
    print('TF-IDF + LR, chia theo bài gốc:', round(tfidf_acc(folds), 3))
    print('TF-IDF + LR, chia ngẫu nhiên:  ', round(tfidf_acc(rand_folds), 3))

## 4. Phân cụm khi không có nhãn (Tuần 10)

### 4.1 Khi không có nhãn
Trong Tuần 9, tâm lớp được tính bằng cách lấy trung bình các bản ghi **có nhãn**. Ở đây ta bỏ nhãn đi. Lúc này không thể tính µ0 và µ1 trực tiếp, vì không biết bản ghi nào thuộc lớp nào. Phân cụm giải quyết bằng cách lặp: đoán tâm, gán điểm vào tâm gần nhất, rồi cập nhật tâm.

In [ ]:
X = np.vstack([class0, class1])   # 8 điểm, nhãn đã bị bỏ đi
print("Hình dạng dữ liệu không nhãn:", X.shape)

### 4.2 K-means tự viết
Mỗi vòng lặp có hai bước, đều dùng lại công thức của Tuần 9:
- **Gán:** mỗi điểm vào tâm có khoảng cách bình phương nhỏ nhất.
- **Cập nhật:** mỗi tâm trở thành trung bình các điểm được gán vào nó (cùng phép trung bình theo từng đặc trưng như slide 8–9 Tuần 9).

In [ ]:
def kmeans_scratch(X, centres, max_iter=100):
    centres = np.array(centres, dtype=float)
    for it in range(1, max_iter + 1):
        d = np.stack([sq_dist(X, c) for c in centres], axis=1)   # n × k
        labels = d.argmin(axis=1)
        new = np.array([X[labels == k].mean(axis=0) if np.any(labels == k) else centres[k]
                        for k in range(len(centres))])
        if np.allclose(new, centres):
            break
        centres = new
    return labels, centres, it

# Khởi tạo: một điểm từ mỗi nhóm (chỉ để minh hoạ, không dùng nhãn)
labels, centres, n_it = kmeans_scratch(X, centres=[[4, 3], [12, 11]])
print("Số vòng lặp:", n_it)
print("Tâm cuối:\n", centres)
# Kiểm tra: phân hoạch phải khớp với nhóm thật (D1..D4 một cụm, P1..P4 một cụm)
assert len(set(labels[:4])) == 1 and len(set(labels[4:])) == 1 and labels[0] != labels[4]
assert np.allclose(centres[labels[0]], mu0) and np.allclose(centres[labels[4]], mu1)
print("K-means tự viết khôi phục đúng hai nhóm.")

Kiểm tra lại bằng thư viện chuẩn. Tên nhãn cụm (0 hoặc 1) là tuỳ ý, nên ta chỉ so sánh cách nhóm, không so sánh số.

In [ ]:
init = np.array([[4, 3], [12, 11]], dtype=float)
km = KMeans(n_clusters=2, init=init, n_init=1, random_state=0).fit(X)
same_partition = (km.labels_[:4] == km.labels_[0]).all() and (km.labels_[4:] == km.labels_[4]).all() \
                 and km.labels_[0] != km.labels_[4]
print("sklearn KMeans cho cùng một phân hoạch:", bool(same_partition))

### 4.3 Thang đo và chuẩn hoá
Khoảng cách phụ thuộc vào đơn vị của đặc trưng (slide 11–12 Tuần 9). Với hai điểm số cùng thang 0–20, chuẩn hoá không đổi nhiều. Phần dưới đây thêm một đặc trưng không liên quan là số phút học (đơn vị khác, độ lớn khác) để thấy đặc trưng có thang lớn sẽ chi phối khoảng cách.

Quy tắc của Tuần 9 vẫn áp dụng: chuẩn hoá phải học từ dữ liệu huấn luyện. Trong phân cụm không có nhãn, "huấn luyện" là toàn bộ tập không nhãn. Không dùng nhãn để chọn cách chuẩn hoá.

In [ ]:
minutes = rng.normal(120, 40, size=len(X))          # đặc trưng nhiễu, đơn vị phút
X3 = np.column_stack([X, minutes])

share = X3.var(axis=0) / X3.var(axis=0).sum()
print("Tỉ lệ phương sai theo từng đặc trưng (trước chuẩn hoá):", share.round(3))

scaler = StandardScaler().fit(X3)                   # fit trên dữ liệu không nhãn
X3s = scaler.transform(X3)
print("Sau chuẩn hoá, trung bình ≈ 0 và độ lệch chuẩn ≈ 1 cho mỗi cột.")
print(X3s.mean(axis=0).round(6), X3s.std(axis=0).round(6))

### 4.4 Chọn số cụm k
Không có nhãn nên không tính được accuracy. Ta dùng hai chỉ số nội tại:
- **Inertia:** tổng khoảng cách bình phương từ điểm đến tâm cụm của nó. Giảm khi k tăng; tìm chỗ "khuỷu tay".
- **Silhouette:** đo từng điểm gần cụm của nó hơn các cụm khác bao nhiêu, trong khoảng −1 đến 1. Càng cao càng tốt.

Dữ liệu tổng hợp dưới đây có ba cụm thật, nhưng hai chỉ số nội tại không nhất thiết chọn đúng 3. Silhouette cao nhất ở k = 2 vì hai cụm thật có tâm gần nhau. Đây là kết quả cần báo cáo và giải thích, không phải lỗi code; ta không chọn k theo nhãn thật.

In [ ]:
means_true = np.array([[7, 6], [15, 14], [12, 4]], dtype=float)
covs_true = [np.array([[5, 1], [1, 5]]), np.array([[4, -1], [-1, 6]]), np.array([[3, 0], [0, 3]])]
sizes = [60, 60, 40]
parts = [rng.multivariate_normal(m, c, size=s) for m, c, s in zip(means_true, covs_true, sizes)]
Xs = np.clip(np.vstack(parts), 0, 20)
y_hidden = np.repeat([0, 1, 2], sizes)              # nhãn thật: chỉ dùng ở mục 7

rows = []
for k in range(2, 7):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    rows.append(dict(k=k, inertia=m.inertia_, silhouette=silhouette_score(Xs, m.labels_)))
kdf = pd.DataFrame(rows)
display(kdf.round(3))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(kdf.k, kdf.inertia, "o-"); ax[0].set_title("Inertia theo k"); ax[0].set_xlabel("k")
ax[1].plot(kdf.k, kdf.silhouette, "o-"); ax[1].set_title("Silhouette theo k"); ax[1].set_xlabel("k")
plt.tight_layout(); plt.show()

### 4.5 Gaussian Mixture: nối với phân phối Gauss
K-means gán cứng: mỗi điểm thuộc đúng một cụm. Gaussian Mixture (GMM) mô hình mỗi cụm là một phân phối Gauss nhiều chiều với tâm **µ** và hiệp phương sai **Σ** (Tuần 9 Part 1, slide 16 và 28 cho phương sai; Resources mục 2 cho Σ). Kết quả là **xác suất** thuộc từng cụm, không chỉ nhãn.

Hiệp phương sai Σ có hai ý nghĩa: đường chéo là phương sai từng đặc trưng; góc ngoài đường chéo cho biết hai đặc trưng biến thiên cùng nhau thế nào (Week 9 Part 2, slide 13). Các trục của elip đồng mức là vector riêng của Σ. Đây là lý do nên làm bài tập trong Resources mục 2 trước khi đọc tiếp.

In [ ]:
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=0).fit(Xs)
proba = gmm.predict_proba(Xs)
print("Xác suất thuộc từng cụm của 3 điểm đầu:")
print(proba[:3].round(3))

# Trục chính của cụm đầu tiên: vector riêng của Σ
evals, evecs = np.linalg.eigh(gmm.covariances_[0])
print("Trị riêng:", evals.round(3))
print("Vector riêng (mỗi cột là một trục):\n", evecs.round(3))

Hình dưới so sánh gán cứng của K-means với xác suất của GMM. Điểm gần ranh giới có xác suất trộn lẫn; K-means không cho thấy điều đó.

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xs)
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
ax[0].scatter(Xs[:, 0], Xs[:, 1], c=km3.labels_, cmap="tab10", s=18)
ax[0].scatter(*km3.cluster_centers_.T, marker="X", s=200, c="k")
ax[0].set_title("K-means: gán cứng")
ax[1].scatter(Xs[:, 0], Xs[:, 1], c=proba.max(axis=1), cmap="viridis", s=18)
ax[1].set_title("GMM: độ chắc chắn (xác suất lớn nhất)")
for a in ax: a.set_xlabel("fractions"); a.set_ylabel("decimals")
plt.tight_layout(); plt.show()

### 4.6 Kiểm tra bằng nhãn ẩn
Nhãn thật (`y_hidden`) chỉ dùng để **kiểm tra** sau khi phân cụm xong, không dùng để huấn luyện. Có hai điều cần lưu ý:
- Tên cụm là tuỳ ý (cụm 0 của K-means có thể là nhãn thật 2), nên không so sánh trực tiếp từng số.
- Adjusted Rand Index (ARI) không phụ thuộc cách đặt tên cụm. ARI bằng 1 khi hai cách nhóm giống hệt nhau và gần 0 khi giống ngẫu nhiên.

In [ ]:
ari_km = adjusted_rand_score(y_hidden, km3.labels_)
ari_gmm = adjusted_rand_score(y_hidden, gmm.predict(Xs))
print(f"ARI K-means = {ari_km:.3f}, ARI GMM = {ari_gmm:.3f}")

# Bảng đối chiếu: mỗi cụm K-means nằm chủ yếu ở nhãn thật nào
table = pd.crosstab(pd.Series(km3.labels_, name="cụm K-means"),
                    pd.Series(y_hidden, name="nhãn thật (ẩn)"))
display(table)

Lưu ý: ARI và bảng đối chiếu chỉ là kiểm tra sau khi phân cụm. Trong một bài toán thật, không có nhãn để so sánh, nên đây không thể là căn cứ duy nhất để kết luận. Giáo viên luôn là người quyết định cụm có ý nghĩa gì.

### 4.7 Dữ liệu thiếu
Từ Tuần 9 (slide 20–21): **không có phản hồi không phải là điểm 0**. Không được điền 0 mặc định. sklearn KMeans không nhận NaN nên phải xử lý trước. Hai cách chấp nhận được: loại bản ghi thiếu, hoặc điền bằng trung bình của **giá trị quan sát được** trên tập huấn luyện. Trong ví dụ dưới đây, ta kiểm tra số bản ghi bị ảnh hưởng và không điền 0.

In [ ]:
Xmiss = Xs.copy()
Xmiss[[3, 17], 1] = np.nan                  # hai bản ghi thiếu điểm decimals
print("Số giá trị thiếu:", int(np.isnan(Xmiss).sum()))

try:
    KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xmiss)
except ValueError as e:
    print("KMeans báo lỗi như dự kiến:", str(e).splitlines()[0])

# Cách 1: loại bản ghi thiếu
keep = ~np.isnan(Xmiss).any(axis=1)
km_drop = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xmiss[keep])
print("Sau khi loại:", keep.sum(), "bản ghi")

# Cách 2: điền trung bình quan sát được (tính trên dữ liệu không thiếu)
col_mean = np.nanmean(Xmiss, axis=0)
Ximp = np.where(np.isnan(Xmiss), col_mean, Xmiss)
print("Trung bình dùng để điền decimals:", col_mean[1].round(3), "(không phải 0)")

## 5. Phân cụm trên dự án OneStopEnglish

In [ ]:
if 'df' not in globals() or 'EXP' not in globals():
    print('Bỏ qua: cần df và EXP từ Group07_Notebook.ipynb. Mở notebook đó, chạy đến hết mục 7 rồi chạy lại ô này.')
else:
    assert df[EXP].isna().sum().sum() == 0, 'Có giá trị thiếu: xử lý trước khi phân cụm, không điền 0.'
    Xu = StandardScaler().fit_transform(df[EXP].values)   # chuẩn hoá không dùng nhãn

    rows = []
    for k in range(2, 6):
        m = KMeans(k, n_init=10, random_state=0).fit(Xu)
        rows.append(dict(k=k, inertia=m.inertia_, silhouette=silhouette_score(Xu, m.labels_)))
    display(pd.DataFrame(rows).round(3))

In [ ]:
if 'df' in globals() and 'EXP' in globals():
    km_p = KMeans(3, n_init=10, random_state=0).fit(Xu)
    gm_p = GaussianMixture(3, covariance_type='full', random_state=0).fit(Xu)
    # Chỉ để kiểm tra, không dùng để huấn luyện
    print('ARI K-means =', round(adjusted_rand_score(y, km_p.labels_), 3))
    print('ARI GMM     =', round(adjusted_rand_score(y, gm_p.predict(Xu)), 3))
    display(pd.crosstab(pd.Series(km_p.labels_, name='cụm K-means'),
                        pd.Series(df.level.values, name='mức thật')))
    P_p = gm_p.predict_proba(Xu)
    print('Số văn bản có xác suất lớn nhất < 0,6 (gần ranh giới):', int((P_p.max(axis=1) < 0.6).sum()), 'trên', len(P_p))

Nếu ARI gần 0, các cụm không trùng với mức. Có thể phân cụm theo chủ đề hoặc độ dài văn bản thay vì mức khó. Điều này nhất quán với kết quả TF-IDF ở mục 6 của notebook dự án: các bản cùng bài rất giống nhau về từ vựng. Đây là kiểm tra mô tả; không kết luận được nguyên nhân từ chỉ số này.

## 6. Việc 2: đọc văn bản nhầm
Mục đích: tách lỗi tiền xử lý khỏi khác biệt độ khó. Ô dưới lập danh sách các văn bản nhầm hai mức (out-of-fold, cùng fold đã lưu), đặt WNL Glastonbury lên đầu, và kiểm tra câu của file gốc sau khi bỏ BOM.

In [ ]:
import re
if 'folds' in globals() and 'EXP' in globals():
    X_ = df[EXP].values
    oofP = np.zeros((len(y), 3))
    for f in folds:
        tr, te = np.array(f['train']), np.array(f['test'])
        m = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000)).fit(X_[tr], y[tr])
        oofP[te] = m.predict_proba(X_[te])
    pred = oofP.argmax(axis=1)
    print('Accuracy out-of-fold (LR, 10 đặc trưng):', round((pred == y).mean(), 3))

    far = [i for i in range(len(y)) if abs(pred[i] - y[i]) == 2]
    print('Số văn bản nhầm hai mức:', len(far))
    far = sorted(far, key=lambda i: (df.article.iloc[i] != 'WNL Glastonbury', df.article.iloc[i]))

    def raw_sent_stats(t):
        t = t.replace('\ufeff', '')
        ss = [s for s in re.split(r'(?<=[.!?])\s+', t) if s.strip()]
        return len(ss), round(sum(len(s.split()) for s in ss) / len(ss), 1), max(len(s.split()) for s in ss)

    rows = []
    for i in far[:10]:
        n, avg, mx = raw_sent_stats(texts[i])
        rows.append({'bài': df.article.iloc[i], 'mức thật': df.level.iloc[i],
                     'dự đoán': LEVELS_W10[pred[i]], 'số câu (regex)': n,
                     'từ/câu (regex)': avg, 'câu dài nhất': mx})
    display(pd.DataFrame(rows))
    print('Văn bản đầu tiên cần đọc:', df.article.iloc[far[0]], '-', df.level.iloc[far[0]])
    print(texts[far[0]][:400].replace('\n', ' / '))

**Mẫu ghi chú đọc (điền ít nhất 10 dòng, bắt đầu từ WNL Glastonbury).** Cột cuối là kết luận của nhóm sau khi đọc, không phải dự đoán của mô hình.

| Bài | Mức thật | Dự đoán | Có lỗi tiền xử lý không? (tiêu đề, BOM, tách câu sai) | Khác biệt do độ khó? (từ vựng, ý, cấu trúc) | Ghi chú |
|---|---|---|---|---|---|
| WNL Glastonbury | Elementary | Advanced | Bản nháp: kiểm tra regex cho khoảng 17 từ mỗi câu, không phải lỗi tách câu. Nhóm cần xác nhận sau khi đọc. | Bản nháp: chưa đọc, nghi do từ vựng. Nhóm xác nhận sau khi đọc. | Điền sau khi đọc |
| | | | | | |

Khi đọc, tự hỏi: câu dài trong bản Elementary có thật không, hay do file có tiêu đề hoặc ký tự lỗi làm câu trông dài hơn? Đây là câu hỏi số 2 ở mục 10.

## 7. Hướng mở rộng: xếp hạng trong bài và gán mức tuyệt đối
Mỗi bài có đủ ba phiên bản. Hai câu hỏi đặt cạnh nhau: (a) mô hình xếp đúng thứ tự ba phiên bản của một bài bao nhiêu phần trăm; (b) khi nhìn một văn bản riêng lẻ, nó gán đúng mức tuyệt đối bao nhiêu phần trăm. Tỉ lệ bộ ba xếp đúng thứ tự (RA) theo định nghĩa trong Pan và cộng sự (2025): tỉ lệ bộ ba mà điểm dự đoán của cả ba phiên bản được xếp đúng. Cần đọc lại định nghĩa trong bài gốc trước khi trích.

In [ ]:
if 'oofP' in globals():
    E = oofP @ np.arange(3)                         # điểm xếp hạng: kỳ vọng mức
    art_idx = {}
    for i, (a, l) in enumerate(zip(df.article, df.level)):
        art_idx.setdefault(a, {})[ORD_W10[l]] = i
    triples = [d for d in art_idx.values() if len(d) == 3]
    ra = sum(E[d[0]] < E[d[1]] < E[d[2]] for d in triples) / len(triples)
    pair_ok = sum(E[d[a]] < E[d[b]] for d in triples for a, b in [(0, 1), (1, 2), (0, 2)]) / (3 * len(triples))
    abs_acc = (pred == y).mean()
    print(f'Bộ ba xếp đúng thứ tự (RA): {ra:.3f} trên {len(triples)} bài')
    print(f'Cặp xếp đúng thứ tự:        {pair_ok:.3f}')
    print(f'Accuracy gán mức tuyệt đối: {abs_acc:.3f}')
    print('Chênh lệch giữa hai con số cho biết độ khó tương đối hay tuyệt đối (xem nhận xét thầy).')

## 8. Việc 3: năm văn bản ngoài bộ dữ liệu
Nhãn phải được nhóm gán **độc lập, trước khi chạy mô hình**. Ghi nhãn vào `external_texts_labels.csv` (cột: `name`, `file`, `label`, `label_date`), đặt nội dung trong thư mục `external_texts/`. Ô dưới đọc nhãn đã ghi, rồi mới chạy dự đoán. Code không kiểm tra được việc gán độc lập; đó là quy trình của nhóm.

In [ ]:
EXT_FILE, EXT_DIR = 'external_texts_labels.csv', 'external_texts'
if 'df' not in globals() or 'EXP' not in globals() or 'extract' not in globals():
    print('Bỏ qua: cần df, EXP và hàm extract() từ notebook dự án.')
elif not os.path.exists(EXT_FILE):
    print('Chưa có', EXT_FILE, '. Tạo file với 5 dòng và điền nhãn trước khi chạy ô này.')
else:
    ext = pd.read_csv(EXT_FILE)
    assert len(ext) == 5, 'Cần đúng 5 văn bản'
    assert ext.label.notna().all() and ext.label_date.notna().all(), 'Thiếu nhãn hoặc ngày gán'
    assert set(ext.label) <= set(LEVELS_W10), 'Nhãn phải là Elementary, Intermediate hoặc Advanced'
    final = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000)).fit(X_, y)
    out = []
    for _, r in ext.iterrows():
        t = open(os.path.join(EXT_DIR, r.file), encoding='utf-8').read()
        f_ = pd.DataFrame([extract(t)])[EXP].values
        p = final.predict_proba(f_)[0]
        out.append({'văn bản': r['name'], 'nhãn nhóm': r.label, 'ngày gán': r.label_date,
                    'dự đoán': LEVELS_W10[int(p.argmax())], 'xác suất lớn nhất': round(p.max(), 3)})
    display(pd.DataFrame(out))

## 9. Trích dẫn và tài liệu
- Vajjala, S. & Lučić, I. (2018). OneStopEnglish corpus: A new corpus for automatic readability assessment and text simplification. *Proceedings of the 13th Workshop on Innovative Use of NLP for Building Educational Applications (BEA)*, trang 297–304. Đã xác minh DOI 10.18653/v1/W18-0535.
- Lee, B. W., Jang, Y. S. & Lee, J. (2021). Pushing on text readability assessment: A transformer meets handcrafted linguistic features. *Proceedings of EMNLP 2021*, trang 10669–10686. DOI 10.18653/v1/2021.emnlp-main.834. **Đã xác minh thông tin thư mục.** Cách chia dữ liệu của bài: **chưa xác minh**; cần đọc mục thí nghiệm của bài gốc và ghi rõ họ chia thế nào.
- Pan, W., Li, X., Chen, X. & Xu, R. (2025). Textual form features for text readability assessment. *Natural Language Processing*, 31, trang 800–841. DOI 10.1017/nlp.2024.50. **Đã xác minh thông tin thư mục.** Định nghĩa RA ở mục 7 cần kiểm tra trong bài gốc.
- Guo, C., Pleiss, G., Sun, Y. & Weinberger, K. Q. (2017). On calibration of modern neural networks. *ICML*. (Dùng ở notebook W9.)

## 10. Bài tập và câu hỏi

**Câu hỏi của thầy**

**1. Nếu chia ngẫu nhiên mà accuracy vẫn bằng đoán bừa, các em sẽ kiểm tra bước nào đầu tiên?**
Trả lời: Bước đầu tiên là căn chỉnh dữ liệu: in vài mẫu ID–text–label, kiểm X, y, groups cùng thứ tự và fold áp đúng X, y. Nếu căn chỉnh đúng mà accuracy vẫn gần 1/3, bước tiếp theo là đo láng giềng gần nhất (mục 3): nếu phần lớn là bản anh em khác mức, nguyên nhân là hiệu ứng cùng bài, không phải lỗi mã.

**2. Một bản Elementary có câu dài như bản Advanced thì điều gì khác khiến người viết lại vẫn xếp nó là Elementary?**
Trả lời: Độ khó không chỉ do độ dài câu. Bản Elementary thường dùng từ thông dụng (tần suất Zipf cao), ít mệnh đề phụ và ý đơn giản hơn, dù câu dài. Vì vậy cần nhìn cả từ vựng và cấu trúc, không chỉ sent_len. Đây cũng là lý do một đặc trưng đơn lẻ không đủ.

**Ba bản của cùng một bài nằm ở cả train và test thì câu hỏi khái quát đã thay đổi thế nào?**
Trả lời: Mô hình được đánh giá trên bài nó đã thấy, nên accuracy đo khả năng nhớ từ vựng của bài chứ không đo khả năng đọc độ khó. Khi chia theo bài, mỗi bài chỉ nằm một phía, nên kết quả phản ánh khả năng khái quát sang bài mới.

**Bài tập Tuần 10** (làm trên giấy trước, rồi chạy ô 44)
- **Bài 1 (K-means bằng tay):** tính khoảng cách bình phương, gán cụm, tính tâm mới với tâm khởi tạo (5, 5) và (15, 15). Kết quả một vòng và kết quả hội tụ được in ở ô 44.
- **Bài 2 (chọn k):** n_init=1 cho inertia lớn hơn hoặc bằng n_init=10. Lý do: n_init=1 chỉ có một lần khởi tạo, có thể rơi vào cực trị cục bộ; n_init=10 lấy kết quả tốt nhất trong mười lần khởi tạo.
- **Bài 3 (ARI):** ARI không đổi khi đổi tên cụm, vì ARI chỉ xét từng cặp điểm có cùng cụm hay không, không phụ thuộc vào số nhãn. Kết quả in ở ô 44 xác nhận điều này.
- **Bài 4 (giải thích):** K-means gán cứng, chỉ dựa vào khoảng cách tới tâm, nên không cho biết điểm nào chắc chắn. GMM có ma trận hiệp phương sai Σ và cho xác suất thuộc từng cụm, nên thấy được các điểm gần ranh giới. Cụm K-means luôn có dạng gần cầu, còn cụm GMM có thể là elip theo hướng của Σ.

In [ ]:
# Bài 1: K-means bằng tay, tâm khởi tạo (5,5) và (15,15)
init1 = np.array([[5, 5], [15, 15]], dtype=float)
D = np.stack([sq_dist(X, c) for c in init1], axis=1)   # khoảng cách bình phương, 8 x 2
print('Khoảng cách bình phương:\n', D)
g1 = D.argmin(axis=1)
print('Gán cụm sau một vòng:', g1)
print('Tâm mới sau một vòng:\n', np.array([X[g1 == k].mean(axis=0) for k in range(2)]))
labels1, centres1, it1 = kmeans_scratch(X, centres=init1)
print('Số vòng đến hội tụ:', it1, '| tâm cuối:', centres1.tolist())

# Bài 2: so sánh inertia giữa n_init=1 và n_init=10 (cùng k=3)
for n in [1, 10]:
    m = KMeans(n_clusters=3, n_init=n, random_state=0).fit(Xs)
    print(f'n_init={n}: inertia={m.inertia_:.3f}')

# Bài 3: đổi tên cụm (0↔2) rồi tính lại ARI
relabelled = np.array([2, 1, 0])[km3.labels_]
print('ARI gốc:', round(adjusted_rand_score(y_hidden, km3.labels_), 3))
print('ARI sau khi đổi tên cụm:', round(adjusted_rand_score(y_hidden, relabelled), 3))

## 11. Kiểm tra trước khi nộp
- [ ] Mục 2: hai bảng cùng thứ tự hàng; in được mẫu ID–text–label.
- [ ] Mục 2: `folds_w10.json` đã lưu; chạy lại vẫn khớp.
- [ ] Mục 3: đã đọc lại và ghi lý do chênh lệch 0,841 và 0,336, kèm bằng chứng.
- [ ] Mục 6: có ít nhất 10 dòng ghi chú đọc, bắt đầu từ WNL Glastonbury.
- [ ] Mục 7: có RA và accuracy tuyệt đối, cùng một bộ fold.
- [ ] Mục 8: đủ 5 văn bản, nhãn có ngày gán, gán trước khi chạy.
- [ ] Mục 9: hai trích dẫn đã xác minh; Lee et al. cần thêm cách chia dữ liệu.
- [ ] Mục 10: hai câu hỏi của thầy đã trả lời.

## 12. Kết luận
- Phân cụm học cấu trúc từ dữ liệu không nhãn bằng cách lặp gán và cập nhật tâm. Các công thức khoảng cách và trung bình giống Tuần 9.
- Chọn k bằng inertia và silhouette, không bằng accuracy vì không có nhãn.
- Thang đo ảnh hưởng đến khoảng cách; chuẩn hoá học từ dữ liệu, không dùng nhãn.
- GMM mô hình hoá từng cụm bằng µ và Σ, cho xác suất thay vì nhãn cứng. Đây là chỗ hiệp phương sai và vector riêng xuất hiện.
- Dữ liệu thiếu không được điền 0. Phải xử lý rõ ràng trước khi phân cụm.
- Cụm chỉ là gợi ý. Giáo viên kiểm tra bản ghi và bối cảnh trước khi đặt tên hoặc đưa ra quyết định.

**Câu hỏi để tự trả lời:** Nếu hai cụm có tâm gần nhau và chỉ số silhouette thấp, điều đó nói lên gì về dữ liệu, và bạn sẽ kiểm tra bước nào tiếp theo?

**Trả lời câu hỏi tự trả lời:** Hai cụm có tâm gần nhau và silhouette thấp cho thấy dữ liệu chồng lấn, không có ranh giới rõ. Điều đó không chứng minh có đúng hai nhóm thật. Bước tiếp theo là đọc lại bản ghi và bối cảnh của dữ liệu trước khi quyết định có tách cụm hay không.